# Chapter 30: Provenance, Acquisition, and Data Quality

This notebook profiles a small NRG shipment snapshot before any repair is attempted.


In [1]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from datasciencebook.provenance_quality import (
    missing_rate, duplicate_keys, range_violations,
    chronological_violations, join_reconciliation, age_hours, quality_gate,
)


## 1. Profile completeness, uniqueness, and validity


In [2]:
shipment_ids = ['S1', 'S2', 'S2', 'S3', 'S4']
temperatures = [24.1, None, 25.0, '', 61.0]
weights = [850, 910, 910, -4, 1200]
print(f'Temperature missing rate: {missing_rate(temperatures):.0%}')
print('Duplicate shipment IDs:', duplicate_keys(shipment_ids))
print('Invalid weight rows:', range_violations(weights, minimum=0, maximum=30_000))


Temperature missing rate: 40%
Duplicate shipment IDs: {'S2': 2}
Invalid weight rows: [3]


## 2. Reconcile a join

The output makes unmatched and duplicate keys visible before the tables are combined.


In [3]:
report = join_reconciliation(
    ['S1', 'S2', 'S2', 'S3', 'S4'],
    ['S1', 'S2', 'S3', 'R5'],
)
for name, value in report.items():
    print(f'{name}: {value}')


left_rows: 5
right_rows: 4
matched_unique_keys: 3
left_only_keys: ['S4']
right_only_keys: ['R5']
duplicate_left_keys: {'S2': 2}
duplicate_right_keys: {}


## 3. Check event order and freshness


In [4]:
dispatch = [datetime(2026, 8, 29, 8), datetime(2026, 8, 29, 10)]
delivery = [datetime(2026, 8, 29, 14), datetime(2026, 8, 29, 9)]
checked_at = datetime(2026, 8, 30, 8)
extracted_at = checked_at - timedelta(hours=25)
print('Chronology violations:', chronological_violations(dispatch, delivery))
print(f'Extract age: {age_hours(extracted_at, checked_at):.0f} hours')


Chronology violations: [1]
Extract age: 25 hours


## 4. Compare missingness by warehouse

The overall rate alone would conceal the concentration in Warehouse C.


In [5]:
warehouse_rates = {'A': 0.02, 'B': 0.04, 'C': 0.31}
plt.bar(list(warehouse_rates), [100 * rate for rate in warehouse_rates.values()], color=['#4472C4', '#70AD47', '#C55A11'])
plt.axhline(5, color='black', linestyle='--', label='5% warning threshold')
plt.ylabel('Missing temperature (%)')
plt.title('NRG temperature missingness by warehouse')
plt.legend()
plt.tight_layout()
plt.show()


<Figure size 640x480 with 1 Axes>

## 5. Apply an explicit quality gate


In [6]:
gate = quality_gate({
    'shipment_id_unique': not duplicate_keys(shipment_ids),
    'weight_valid': not range_violations(weights, minimum=0, maximum=30_000),
    'timestamps_ordered': not chronological_violations(dispatch, delivery),
    'extract_under_26_hours': age_hours(extracted_at, checked_at) < 26,
})
print(gate)


{'passed': False, 'failed_rules': ['shipment_id_unique', 'weight_valid', 'timestamps_ordered']}


## Practice

Add a route-domain rule and document its denominator, scope, severity, and response.


In [ ]:
# Write and evaluate your practice rule here.
